In [ ]:
# Cell 1: Install
!pip install ultralytics opencv-python matplotlib

In [ ]:
# Cell 2: Imports
from ultralytics import YOLO
import cv2
import os, glob, shutil, random, collections
import matplotlib.pyplot as plt

In [ ]:
# Cell 3: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 4: Create labels (skips images that already have a label file)
BASE = "/content/drive/MyDrive/dog_cat_dataset/dog_cat"

det = YOLO("yolov8x.pt")   # COCO: cat = 15, dog = 16

for split in ["train", "test"]:
    root = f"{BASE}/{split}"
    os.makedirs(f"{root}/labels", exist_ok=True)
    for p in glob.glob(f"{root}/images/*"):
        stem, ext = os.path.splitext(os.path.basename(p))
        out = f"{root}/labels/{stem}.txt"
        if ext.lower() not in (".jpg", ".jpeg", ".png") or os.path.exists(out):
            continue
        r = det(p, classes=[15, 16], verbose=False)[0]
        if len(r.boxes) == 0:
            continue
        i = int(r.boxes.conf.argmax())
        name = stem.lower()
        if "cat" in name and "dog" not in name:
            cls = 0
        elif "dog" in name and "cat" not in name:
            cls = 1
        else:                                   # unclear filename: use the detector's class
            cls = 0 if int(r.boxes.cls[i]) == 15 else 1
        x, y, w, h = r.boxes.xywhn[i].tolist()
        open(out, "w").write(f"{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")

c = collections.Counter()
for f in glob.glob(f"{BASE}/train/labels/*.txt"):
    c[open(f).read().split()[0]] += 1
print("train labels:", sum(c.values()), "| cat(0):", c["0"], "| dog(1):", c["1"])

In [ ]:
# Cell 5: Prepare data + data.yaml
BASE = "/content/drive/MyDrive/dog_cat_dataset/dog_cat"
LOCAL = "/content/dog_cat"

shutil.rmtree(LOCAL, ignore_errors=True)
shutil.copytree(BASE, LOCAL, ignore=shutil.ignore_patterns("*.cache", "data.yaml"))
os.makedirs(f"{LOCAL}/valid/images"); os.makedirs(f"{LOCAL}/valid/labels")
random.seed(42)
imgs = sorted(glob.glob(f"{LOCAL}/train/images/*"))
for p in random.sample(imgs, int(0.15 * len(imgs))):
    stem = os.path.splitext(os.path.basename(p))[0]
    shutil.move(p, f"{LOCAL}/valid/images/")
    lbl = f"{LOCAL}/train/labels/{stem}.txt"
    if os.path.exists(lbl):
        shutil.move(lbl, f"{LOCAL}/valid/labels/")

data_yaml = f"""
path: {LOCAL}
train: train/images
val: valid/images

nc: 2

names: ['cat', 'dog']
"""

with open(f"{LOCAL}/data.yaml", "w") as f:
    f.write(data_yaml)

print("data.yaml file created successfully")
print("train labels:", len(glob.glob(f"{LOCAL}/train/labels/*.txt")))   # must be above 0

In [ ]:
# Cell 6: Train
model = YOLO("yolov8n.pt")

model.train(
    data=f"{LOCAL}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=8,
    name="dog_cat_detector"
)

In [ ]:
# Cell 7: Load the trained model and save it to Drive
from ultralytics import YOLO
import os, glob, shutil

weights = glob.glob("/content/runs/**/best.pt", recursive=True)
if not weights:
    weights = glob.glob("/content/runs/**/last.pt", recursive=True)

drive_copy = "/content/drive/MyDrive/best.pt"
if not weights and os.path.exists(drive_copy):
    weights = [drive_copy]

if not weights:
    print("No trained weights found. Cell 6 (Train) did not finish.")
    print("local train labels:", len(glob.glob("/content/dog_cat/train/labels/*.txt")))
    print("Fix: re-run cell 4, then cell 5, then cell 6 and wait for 'Results saved to ...'")
else:
    best = max(weights, key=os.path.getmtime)
    print("using:", best)
    trained_model = YOLO(best)
    if best != drive_copy:
        shutil.copy(best, drive_copy)
        print("saved to Drive:", drive_copy)

In [ ]:
# Cell 8: Show detection images (test folder, else valid; max 10 images)
import cv2, os, glob
import matplotlib.pyplot as plt

LOCAL = "/content/dog_cat"

files = sorted(glob.glob(f"{LOCAL}/test/images/*"))
if not files:
    files = sorted(glob.glob(f"{LOCAL}/valid/images/*"))
MAX_IMAGES = 10
if len(files) < MAX_IMAGES:
    files += sorted(glob.glob(f"{LOCAL}/train/images/*"))
files = files[:MAX_IMAGES]   # show at most 10 images

for path in files:
    image = cv2.imread(path)
    if image is None:
        continue
    results = trained_model(image, conf=0.25, verbose=False)
    annotated_frame = cv2.cvtColor(results[0].plot(), cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 8))
    plt.imshow(annotated_frame)
    plt.title(f"Detection Result: {os.path.basename(path)}")
    plt.axis("off")
    plt.show()